# exp084_pf_confidence_residual_clip train

Conservative OOF postprocess guard for the deterministic exp073 full replay ML anchor. This notebook also renders fold-averaged LightGBM feature importance from the exp073 saved boosters.

## Contents

1. Setup and configuration
2. Exp073 OOF and saved model sources
3. Postprocess guard audit
4. Fold-averaged feature importance
5. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

from settings import ExperimentPaths, load_config, get_nested
from exp063_full_replay_reproducibility_guard import (
    OUTPUT_PREFIX,
    POSTPROCESS_PREFIX,
    find_exp073_train_artifact,
    find_model_manifest,
    run_postprocess_guard,
    run_saved_model_feature_importance_audit,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "audit.mode"))
print("Parent:", cfg_get(config, "lineage.parent"))
print("Kernel sources:", cfg_get(config, "runtime.kaggle.kernel_sources"))
print("Selected source:", cfg_get(config, "audit.selected_mode"), cfg_get(config, "audit.selected_model"))


## 2. Exp073 OOF and saved model sources

In [ ]:
oof_path = find_exp073_train_artifact(
    f"{OUTPUT_PREFIX}_predictions.csv.gz",
    cfg_get(config, "data.exp073_oof_predictions_local"),
)
manifest_path = find_model_manifest(cfg_get(config, "inference.model_manifest_path"))
print("exp073 OOF predictions:", oof_path)
print("exp073 model manifest:", manifest_path)
preview = pd.read_csv(oof_path, nrows=5, dtype={"id": str, "well": str})
display(preview)


## 3. Postprocess guard audit

In [ ]:
summary = run_postprocess_guard(
    output_dir=paths.artifacts_dir,
    predictions_path=cfg_get(config, "data.exp073_oof_predictions_local"),
    feature_cache_path=cfg_get(config, "data.exp072_train_feature_cache_local"),
    mode_name=str(cfg_get(config, "audit.selected_mode", "gpu_repro_guard_dp_threads8")),
    model_name=str(cfg_get(config, "audit.selected_model", "lgb_mean")),
    clip_quantiles=list(cfg_get(config, "audit.clip_quantiles", [0.99, 0.995, 0.999])),
)
print(json.dumps(summary, indent=2))


## 4. Fold-averaged feature importance

In [ ]:
importance_summary = run_saved_model_feature_importance_audit(
    output_dir=paths.artifacts_dir,
    model_manifest_path=cfg_get(config, "inference.model_manifest_path"),
    mode_name=str(cfg_get(config, "audit.selected_mode", "gpu_repro_guard_dp_threads8")),
    top_n=40,
)
print(json.dumps(importance_summary, indent=2))
plot_name = importance_summary["artifacts"].get("feature_importance_plot")
if plot_name:
    display(Image(filename=str(paths.artifacts_dir / plot_name)))


## 5. Metrics and artifacts

In [ ]:
post_metrics = pd.read_csv(paths.artifacts_dir / f"{POSTPROCESS_PREFIX}_metrics.csv")
post_buckets = pd.read_csv(paths.artifacts_dir / f"{POSTPROCESS_PREFIX}_bucket_metrics.csv")
importance = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_importance_mean.csv")

display(post_metrics.sort_values("rmse_tvt"))
display(post_buckets.sort_values(["policy", "distance_bucket"]).head(80))
display(importance.sort_values("gain_mean", ascending=False).head(40))
print("Artifacts dir:", paths.artifacts_dir)
